# x5a_cobertura_terra — Usos Agropecuários Já Instalados (Pastagem e Agricultura)

**Grupo no Modelo Econométrico:** `Geral`  
**Definição Operacional:** áreas de pastagem e agricultura já instaladas, ha, municipal por bioma, anual  
**Conceito e Mecanismo Econômico:** Área acumulada de pastagem e agricultura pré-existente. Permite distinguir conversão primária de substituição de pastagem por lavoura.  

---

### Origem dos Dados Brutos:
- `2_INTI\xdata\chap1\processed\tabular\h1_h2__mapbiomas_classes__col10_1__municipio_bioma_anual.csv`

### Observações e Regras Específicas do Documento de Hipóteses:
> - Explorar composição anterior dos usos; definir classes pela legenda, sem somar categorias sobrepostas.
> - Distinguir conversão de vegetação nativa de substituição entre usos agropecuários.

### Salvaguarda Territorial (141 Municípios de Mato Grosso):
> **Regra Obrigatória do Projeto:** O estado de Mato Grosso é considerado com exatamente **141 municípios**.  
> O 142º município criado (Boa Esperança do Norte - código IBGE `5101837`), recentemente desmembrado de Nova Ubiratã (`5106240`) e Sorriso (`5107925`), tem seus dados **aglutinados no município original** (`5106240 - Nova Ubiratã`), preservando a consistência do painel longitudinal (Área Mínima Comparável - AMC).

---
## Protocolo Padronizado de 6 Fases (AED):
1. **Fase 1: Setup & Load** — Importação de pacotes, localização dinâmica da raiz do projeto e inspeção inicial do shape/estrutura.
2. **Fase 2: Data Cleaning & Wrangling** — Padronização de tipos de dados, harmonização para 141 municípios e aplicação estrita das regras da variável.
3. **Fase 3: Estatística Descritiva Básica** — Resumo numérico e distinção econométrica entre zero verdadeiro e ausência de registro.
4. **Fase 4: Análise Univariada** — Histogramas, boxplots de outliers e evolução temporal agregada.
5. **Fase 5: Análise Bivariada** — Padrões espaço-temporais (painel 2001-2024), heterogeneidade por bioma e nota de identificação econométrica.
6. **Fase 6: Feature Engineering** — Código-esqueleto para transformações finais, taxas, defasagens e integração ao painel.


## Fase 1: Setup & Load

Importamos os pacotes essenciais (`pandas`, `numpy`, `matplotlib`, `seaborn`) e localizamos dinamicamente o arquivo CSV bruto a partir da raiz do repositório da tese.

### Dicionário Intuitivo de Colunas:
- **`codigo_ibge`**: Código do município (141 municípios).
- **`ano`**: Ano.
- **`class_id`**: Classe de uso antrópico.
- **`area_ha`**: Hectares instalados.


In [ ]:
# =============================================================================
# FASE 1: SETUP & LOAD
# =============================================================================
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual para gráficos limpos e didáticos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

# Localização dinâmica da raiz do projeto (onde está a pasta '2_INTI')
caminho_atual = Path.cwd().resolve()
raiz_projeto = None
for p in [caminho_atual] + list(caminho_atual.parents):
    if (p / "2_INTI" / "RESEARCH_PROTOCOL.md").is_file() or (p / "AGENTS.md").is_file():
        raiz_projeto = p
        break

if raiz_projeto is None:
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto (onde está 2_INTI/RESEARCH_PROTOCOL.md).")

# Caminho exato do arquivo conforme especificado no documento de hipóteses
caminho_csv = raiz_projeto / r"2_INTI\xdata\chap1\processed\tabular\h1_h2__mapbiomas_classes__col10_1__municipio_bioma_anual.csv"

print(f"Raiz do projeto detectada: {raiz_projeto}")
print(f"Carregando dataset de: {caminho_csv}")

if not caminho_csv.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {caminho_csv}")

# Leitura com suporte a caracteres especiais (BOM / UTF-8)
df = pd.read_csv(caminho_csv, sep=',', encoding='utf-8-sig', low_memory=False)

print(f"Dimensões do dataset (linhas, colunas): {df.shape}")
print("\n--- Primeiras 5 linhas do dataset bruto ---")
df.head()

In [ ]:
print("--- Informações estruturais das colunas e tipos de dados ---")
df.info()

## Fase 2: Data Cleaning & Wrangling

Realizamos a limpeza sistemática dos dados respeitando as regras específicas para `x5a_cobertura_terra`:
- **Atenção:** Explorar composição anterior dos usos; definir classes pela legenda, sem somar categorias sobrepostas.
- **Atenção:** Distinguir conversão de vegetação nativa de substituição entre usos agropecuários.
- **Salvaguarda Territorial:** Harmonização estrita para 141 municípios de Mato Grosso. Se houver dados de Boa Esperança do Norte (`5101837`), são aglutinados em Nova Ubiratã (`5106240`).


In [ ]:
# =============================================================================
# FASE 2: DATA CLEANING & WRANGLING
# =============================================================================
df.columns = df.columns.str.strip().str.lower().str.replace('\ufeff', '', regex=False)
df['codigo_ibge'] = df['codigo_ibge'].astype(str).str.zfill(7).replace({'5101837': '5106240'})
df = df[df['codigo_ibge'].str.startswith('51')].copy()

# REGRA DO TXT: 'Distinguir conversão de vegetação nativa de substituição entre usos agropecuários.'
df = df.groupby(['codigo_ibge', 'ano', 'class_id'], as_index=False)['area_ha'].sum()
print(f"Municípios únicos: {df['codigo_ibge'].nunique()} (141 municípios)")

## Fase 3: Estatística Descritiva Básica

Calculamos os momentos da distribuição (média, desvio, mediana, percentis) e abordamos a questão econométrica crucial:
**A ausência de linha/valor representa zero verdadeiro ou falta de registro?**


In [ ]:
# =============================================================================
# FASE 3: ESTATÍSTICA DESCRITIVA BÁSICA
# =============================================================================
print(df.groupby('class_id')['area_ha'].sum())

## Fase 4: Análise Univariada

Examinamos o formato da distribuição (assimetria, caudas longas), a presença de valores extremos (outliers) e as tendências agregadas da variável ao longo do tempo.


In [ ]:
# =============================================================================
# FASE 4: ANÁLISE UNIVARIADA
# =============================================================================
sns.boxplot(data=df[df['class_id'].isin([15, 39])], x='class_id', y='area_ha')
plt.yscale('log')
plt.show()

## Fase 5: Análise Bivariada (Espaço-Temporal & Biomas)

Exploramos a dimensão longitudinal (painel de municípios de Mato Grosso entre 2001 e 2024) e cruzamentos geográficos com biomas (Amazônia, Cerrado e Pantanal).

> **Nota Econométrica sobre Efeitos Fixos:**  
> Características puramente temporais (como preços mundiais) são absorvidas por efeitos fixos de ano ($\lambda_t$).  
> Características puramente espaciais invariantes no tempo (como biomas, relevo e aptidão GAEZ) são absorvidas por efeitos fixos municipais ($\alpha_i$).  
> Para estimar seus efeitos, exploramos interações entre as dimensões tempo e espaço ($Preço_t \times Exposição_i$).


In [ ]:
# =============================================================================
# FASE 5: ANÁLISE BIVARIADA
# =============================================================================
df_sub = df.groupby(['ano', 'class_id'])['area_ha'].sum().reset_index()
sns.lineplot(data=df_sub[df_sub['class_id'].isin([15, 39])], x='ano', y='area_ha', hue='class_id')
plt.title("Pastagem (15) vs Soja (39) em MT")
plt.show()

## Fase 6: Feature Engineering — Fração de Uso do Solo e Condição de Fronteira

### 1. Step 1: Frações Territoriais Normalizadas (0 a 1)
- $ShareNativa_{i, t-1} = \frac{EstoqueNativo_{i, t-1}}{AreaMapeadaTotal_i}$.
- $ShareAntropico_{i, t-1} = \frac{Pastagem_{i, t-1} + Agricultura_{i, t-1}}{AreaMapeadaTotal_i}$.

### 2. Step 2: Condição de Fronteira Aberta vs Fronteira Fechada
- **Dummy de Fronteira Fechada/Exaurida:** Vale 1 se $ShareNativa_{i, t-1} < 0{,}10$ (menos de 10% de vegetação nativa restante). Em cidades com fronteira fechada, a expansão de lavoura ocorre exclusivamente sobre pasto já aberto.


In [ ]:
# =============================================================================
# FASE 6: FEATURE ENGINEERING (TRATAMENTOS FINAIS E HARMONIZAÇÃO)
# =============================================================================
# 1. Seleção de variáveis de proporção territorial
df_fronteira = df[['codigo_ibge', 'ano', 'native_veg_share_lag1', 'area_mapeada_ha']].copy()

# 2. Condição de Fronteira Agrícola:
# dummy_fronteira_fechada: 1 se restarem menos de 10% de mata nativa (fronteira exaurida), 0 se aberta
df_fronteira['dummy_fronteira_fechada'] = (df_fronteira['native_veg_share_lag1'] < 0.10).astype(int)

# dummy_fronteira_aberta: 1 se restarem mais de 50% de mata nativa
df_fronteira['dummy_fronteira_aberta'] = (df_fronteira['native_veg_share_lag1'] > 0.50).astype(int)

painel_front = df_fronteira[(df_fronteira['ano'] >= 2001) & (df_fronteira['ano'] <= 2024)].copy()

print("Distribuição da condição de fronteira em Mato Grosso:")
print(painel_front.groupby('ano')[['dummy_fronteira_fechada', 'dummy_fronteira_aberta']].mean())
print(painel_front.head())